In [1]:
import torch
import torch.nn as nn 
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PriceDataset(Dataset):
    def __init__(self, img_path, image_paths, item_names, bullet_points, product_descriptions, values, units, prices, processor):
        self.dir = img_path
        self.image_paths = image_paths
        self.item_names = item_names
        self.bullet_points = bullet_points
        self.product_descriptions = product_descriptions
        self.values = values
        self.units = units
        self.prices = torch.tensor(prices, dtype=torch.float32)  # Changed to float32 for regression
        self.processor = processor
    
    def __len__(self):
        return len(self.image_paths)
    
    def _concatenate_text(self, idx):
        """Concatenate all text fields into one string"""
        text_parts = []
        
        # Add Item Name
        if pd.notna(self.item_names[idx]):
            text_parts.append(f"Item Name: {self.item_names[idx]}")
        
        # Add Bullet Points
        if pd.notna(self.bullet_points[idx]):
            text_parts.append(f"Bullet Points: {self.bullet_points[idx]}")
        
        # Add Product Description
        if pd.notna(self.product_descriptions[idx]):
            text_parts.append(f"Product Description: {self.product_descriptions[idx]}")
        
        # Add Value and Unit
        if pd.notna(self.values[idx]) and pd.notna(self.units[idx]):
            text_parts.append(f"Quantity: {self.values[idx]} {self.units[idx]}")
        
        return " ".join(text_parts)
    
    def __getitem__(self, idx):
        path = os.path.join(self.dir, str(self.image_paths[idx]) + '.jpg')
        img = Image.open(path).convert('RGB')
        
        # Concatenate all text fields
        full_text = self._concatenate_text(idx)
        
        inputs = self.processor(
            images=img, 
            text=full_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'price': self.prices[idx]
        }

In [ ]:
class SMAPELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    
    def forward(self, pred, target):
        # Ensure inputs are valid
        pred = torch.clamp(pred, min=-1e6, max=1e6)  # Prevent extreme values
        target = torch.clamp(target, min=-1e6, max=1e6)
        
        numerator = torch.abs(pred - target)
        denominator = (torch.abs(pred) + torch.abs(target) + self.eps)
        
        # Prevent division by zero
        denominator = torch.clamp(denominator, min=self.eps)
        
        smape = 2.0 * numerator / denominator
        
        # Check for NaN/inf before returning
        smape = torch.where(torch.isnan(smape) | torch.isinf(smape), 
                           torch.zeros_like(smape), smape)
        
        return torch.mean(smape) * 100

In [ ]:
class BLIP2PricePredictor(nn.Module):
    def __init__(self, model_name="Salesforce/blip2-opt-2.7b"):
        super().__init__()
        self.blip2 = Blip2ForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        for param in self.blip2.parameters():
            param.requires_grad = False
        
        qformer_dim = self.blip2.config.qformer_config.hidden_size
        vocab_size = self.blip2.config.text_config.vocab_size

        # Get device from the model
        model_device = next(self.blip2.parameters()).device
        
        self.word_emb = torch.nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=qformer_dim,
        )
        # Very conservative initialization
        torch.nn.init.normal_(self.word_emb.weight, mean=0.0, std=0.001)  # Much smaller std
        self.word_emb.to(model_device, dtype=torch.float16)
        
        # Simpler regressor for stability
        self.regressor = nn.Sequential(
            nn.Linear(qformer_dim, 128),  # Smaller hidden size
            nn.LayerNorm(128),  # Add layer norm for stability
            nn.ReLU(),
            nn.Dropout(0.05),  # Less dropout
            nn.Linear(128, 1)
        )
        
        # Very conservative initialization
        for layer in self.regressor:
            if isinstance(layer, nn.Linear):
                torch.nn.init.xavier_normal_(layer.weight, gain=0.01)  # Very small gain
                torch.nn.init.zeros_(layer.bias)
        
        # Keep regressor in float32
        self.regressor.to(model_device, dtype=torch.float32)

    def forward(self, pixel_values, input_ids=None, attention_mask=None, **kwargs):
        # Get image features
        input_ids = torch.clamp(input_ids, min=0, max=50264)  # Clamp to valid vocab range
        vision_outputs = self.blip2.vision_model(pixel_values)
        
        image_embeds = vision_outputs[0]
        del pixel_values, vision_outputs
        
        image_attn_mask = torch.ones(
            image_embeds.size()[:-1],
            dtype=torch.long,
            device=image_embeds.device
        )
        
        # Use learnable query tokens with more conservative approach
        word_embeddings = self.word_emb(input_ids)
        # Take only first few tokens to reduce complexity
        query_tokens = word_embeddings[:, :32]  # Limit sequence length
        
        # Get the actual QFormer model
        if hasattr(self.blip2.qformer, 'base_model'):
            qformer_model = self.blip2.qformer.base_model
        else:
            qformer_model = self.blip2.qformer
        
        # Call QFormer
        query_outputs = qformer_model(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attn_mask,
            return_dict=True,
        )
        
        # Convert to float32 before regressor and add stability
        qformer_out = query_outputs.last_hidden_state.mean(dim=1).float()
        
        # More conservative clamping
        qformer_out = torch.clamp(qformer_out, min=-5, max=5)
        
        price_pred = self.regressor(qformer_out)
        
        return price_pred.squeeze(-1)

In [6]:
def apply_qlora(model, r=16, alpha=32):
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=[
            "query", "key", "value",
            "attention.self.query",
            "attention.self.key", 
            "attention.self.value",
            "crossattention.self.query",
            "crossattention.self.key",
            "crossattention.self.value"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    
    model.blip2.qformer = get_peft_model(model.blip2.qformer, lora_config)
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    
    return model

In [ ]:
def train(model, train_loader, val_loader, epochs=10, lr=1e-6, device='cuda', log_file='training_log.csv'):    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=0.0001,
        eps=1e-8,
        betas=(0.9, 0.999)
    )
    
    criterion = nn.HuberLoss(delta=1.0)
        
    def lr_lambda(step):
        warmup_steps = 100
        if step < warmup_steps:
            return step / warmup_steps
        return 1.0
    
    warmup_scheduler = LambdaLR(optimizer, lr_lambda)
    main_scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    
    accum_steps = 8
    
    # Initialize CSV logging
    if not os.path.exists(log_file):
        with open(log_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Epoch', 'Train_Loss', 'Val_Loss'])
    
    global_step = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        
        # Training loop with tqdm
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False)
        for i, batch in enumerate(train_pbar):
            torch.cuda.empty_cache()
            
            pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
            input_ids = batch['input_ids'].to(device)
            prices = batch['price'].to(device, dtype=torch.float32)
            
            if torch.isnan(pixel_values).any() or torch.isnan(prices).any():
                continue
            
            preds = model(pixel_values, input_ids)
            
            if torch.isnan(preds).any() or torch.isinf(preds).any():
                continue
            
            preds = torch.clamp(preds, min=-5, max=10)
            loss = criterion(preds, prices) / accum_steps
            
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            
            loss.backward()
            del pixel_values, input_ids, preds
            
            if (i + 1) % accum_steps == 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                
                has_nan_grad = False
                for name, param in model.named_parameters():
                    if param.grad is not None and (torch.isnan(param.grad).any() or torch.isinf(param.grad).any()):
                        has_nan_grad = True
                        break
                
                if not has_nan_grad and grad_norm < 10:
                    optimizer.step()
                    
                    if epoch == 0 and global_step < 100:
                        warmup_scheduler.step()
                    
                    global_step += 1
                
                optimizer.zero_grad()
                torch.cuda.empty_cache()
            
            train_loss += loss.item() * accum_steps
            
            # Update tqdm progress bar
            train_pbar.set_postfix({
                'loss': f'{loss.item() * accum_steps:.6f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
            })
            
            if loss.item() > 50:
                return model
        
        train_pbar.close()
        
        # Validation loop with tqdm
        model.eval()
        val_loss = 0.0
        valid_batches = 0
        
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]', leave=False)
        with torch.no_grad():
            for batch in val_pbar:
                pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
                input_ids = batch['input_ids'].to(device)
                prices = batch['price'].to(device, dtype=torch.float32)
                
                preds = model(pixel_values, input_ids)
                
                if not (torch.isnan(preds).any() or torch.isinf(preds).any()):
                    preds = torch.clamp(preds, min=-5, max=10)
                    loss = criterion(preds, prices)
                    
                    if not torch.isnan(loss):
                        val_loss += loss.item()
                        valid_batches += 1
                        val_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
                
                del pixel_values, input_ids, preds
        
        val_pbar.close()
        
        avg_train_loss = train_loss / len(train_loader) if len(train_loader) > 0 else 0
        avg_val_loss = val_loss / valid_batches if valid_batches > 0 else 0
        
        # Log to CSV
        with open(log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, avg_train_loss, avg_val_loss])
        
        # Print epoch summary (this won't break tqdm as it's between epochs)
        tqdm.write(f'Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
        
        if epoch > 0:
            main_scheduler.step()
        
        torch.cuda.empty_cache()
    
    return model

In [ ]:
MODEL = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(MODEL)
TRAIN_IMG_DIR = '../input/amazon-train-set-part1/train_part1/train_part1'
VAL_IMG_DIR = '../input/amazon-train-set-part1/val_part1/val_part1'
train_df = pd.read_csv('../input/amazon-train-set-part1/train_part1.csv')
val_df = pd.read_csv('../input/amazon-train-set-part1/val_part1.csv') 

train_dl = DataLoader(
    PriceDataset(
        img_path = TRAIN_IMG_DIR,
        image_paths=train_df['sample_id'].values,
        item_names=train_df['Item Name'].values,
        bullet_points=train_df['Bullet Points'].values,
        product_descriptions=train_df['Product Description'].values,
        values=train_df['Value'].values,
        units=train_df['Unit'].values,
        prices=train_df['log_price'].values,  
        processor=processor
    ),
    batch_size=8,
    shuffle=True,
    num_workers=4
)
val_dl = DataLoader(
    PriceDataset(
        img_path = VAL_IMG_DIR,
        image_paths=val_df['sample_id'].values,
        item_names=val_df['Item Name'].values,
        bullet_points=val_df['Bullet Points'].values,
        product_descriptions=val_df['Product Description'].values,
        values=val_df['Value'].values,
        units=val_df['Unit'].values,
        prices=val_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=False,
    num_workers=4
)    

model = BLIP2PricePredictor(MODEL)
model = apply_qlora(model, r=16, alpha=32)
    
model = train(model, train_dl, val_dl, epochs=10, lr=3e-6)
torch.save({
        'regressor': model.regressor.state_dict(),
        'qformer': model.blip2.qformer.state_dict(),
    }, 'model.pth')
    
print("Done!")

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.34s/it]
/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the cpu.


Trainable: 25,088,001 / 3,898,628,097 (0.64%)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ~~~~~~~~~~~~~~~^^^^^^
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/torch/utils/data/_utils/collate.py", line 172, in collate
    key: collate(
         ~~~~~~~^
        [d[key] for d in batch], collate_fn_map=collate_fn_map
        ^^^^^^^^^^^^^^^^^^^^^^^^^